# extract

> what a document *is*, the fields inside it, and an answer over the whole of it

In [ ]:
#| default_exp extract

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

Retrieval answers *where* something is. This module answers two questions it cannot: **what kind of
thing** each document in the vault is, and **what fields** are inside it — a vault that knows which
of its documents are invoices can hand you their totals as a table.

Three legs, cheapest first, and each says which one answered:

|leg|needs|what it does|
|---|---|---|
|regex signals|nothing|money, dates, references, tables, code — the numeric evidence a type is scored on|
|noun entities + keyphrases|nothing|`ORG`, `PERSON`, `LAW`… by regex; yake keyphrases as a fallback for the rest|
|a small LLM|`rishi`|the label when the cues cannot decide, and every structured extraction|

The order is the design. Typing ten thousand documents through an LLM is hours of compute to answer
a question a cue table answers for most of them — so the model is spent only where the table knows
it is guessing.

In [ ]:
#| export
import json, re, warnings
from collections import Counter
from dataclasses import dataclass, fields, is_dataclass, make_dataclass, asdict
from typing import get_origin
from fastcore.all import AttrDict, L, patch
from litesearch import text_entities
from rishi.core import Chat, extract_fence, infer_runtime, resp_text
from vishalakshi.core import Vault
from vishalakshi.ask import VAULT_SP, dflt_model, is_stock_chat, new_chat, split_reasoning   # also patches Vault.ask

## Signals: what a document contains

`signals` counts the evidence, and is the one call that every leg of categorisation shares.

The regex leg always runs and owns the numeric evidence — money, dates, percentages, reference
numbers — which is where a cue table gets most of its confidence about paperwork. The entity leg
adds structural labels: `ORG` by legal suffix (`Ltd`, `Corp`, `GmbH`…), `PERSON` by honorific
(`Dr.`, `Prof.`, `Mr.`…), `LAW` by instrument type (`Act`, `Directive`, `Convention`…). Where those
patterns miss, yake keyphrases fill in as `KEYPHRASE`. Labels are kept separately in `labels` so a
scorer asking what the entity leg specifically found is not handed a regex hit wearing the same name,
and `method` says which legs ran, because a score is only comparable against one from the same legs.

In [ ]:
#| export
SIGNALS = dict(
    # what a document *contains*, as cheaply as a regex can tell — the evidence a type is scored on
    money    = r'[$€£¥₹]\s?\d[\d,]*(?:\.\d+)?|\b\d[\d,]*\.\d{2}\s?(?:usd|eur|gbp|inr|jpy|cad|aud)\b'
               r'|\b(?:usd|eur|gbp|inr|jpy|cad|aud)\s?\d[\d,]*',
    date     = r'\b\d{4}-\d{2}-\d{2}\b|\b\d{1,2}[/.]\d{1,2}[/.]\d{2,4}\b'
               r'|\b\d{1,2} (?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]* \d{2,4}\b'
               r'|\b(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]* \d{1,2},? \d{4}\b',
    time     = r'\b\d{1,2}:\d{2}(?::\d{2})?\s?(?:am|pm)?\b',
    ref      = r'\b(?:invoice|inv|order|p\.?o|ref(?:erence)?|receipt|bill|account|sku|part|item)\s*'
               r'(?:no\.?|number|id)?\s*[:#]\s*[a-z0-9][a-z0-9\-/]{2,}\b',
    qty      = r'\b(?:qty|quantity|units?|pcs?|nos?)\b\s*[:.]?\s*\d+|\b\d+\s?(?:x|×)\s?\d',
    percent  = r'\b\d{1,3}(?:\.\d+)?\s?%',
    tax      = r'\b(?:vat|gst|hsn|sac|tin|ein|abn|sales tax|tax id|withholding)\b',
    email    = r'\b[\w.+-]+@[\w-]+\.[\w.]{2,}\b',
    url      = r'https?://\S+',
    citation = r'\[\d+\]|\bet al\.|\bdoi:|\barxiv:',
    code     = r'^\s*(?:def|class|function|const|let|var|import|from|package|#include)\s+\w',
    table    = r'^\s*\|.+\|\s*$',
    heading  = r'^#{1,6} \S',
    blank    = r'_{4,}|\[ ?\]',
)
_SIG = {k: re.compile(v, re.I|re.M) for k, v in SIGNALS.items()}
# entity extraction runs on the head of a document — that is where its type is decided
NER_CHARS = 20000

# handrolled entity extraction: ORG by legal suffix, PERSON by honorific, LAW by instrument type
_ORG_SUFF = re.compile(
    r'(?<![a-z,\.])([A-Z][A-Za-z0-9\'\-& ]{1,50}?)\s+'
    r'(?:Ltd\.?|Limited|Corp\.?|Corporation|Inc\.?|LLC|LLP|GmbH|AG|SA|NV|BV|Pty\.?|PLC'
    r'|Group\b|Holdings?\b|Associates?\b|Partners?\b|Ventures?\b'
    r'|University\b|College\b|Institute\b|Hospital\b|Foundation\b'
    r'|Bank\b|Fund\b|Trust\b|Council\b|Authority\b|Commission\b'
    r'|Ministry\b|Department\b|Agency\b|Bureau\b)\b'
)
_PERSON_HON = re.compile(
    r'\b(?:Mr\.?|Mrs\.?|Ms\.?|Miss|Dr\.?|Prof(?:essor)?\.?|Rev\.?|Sir|Dame|Lord|Lady)\s+'
    r'([A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,2})\b'
)
_LAW_INST = re.compile(
    r'\b([A-Z][A-Za-z ]{2,50}?)\s+'
    r'(?:Act(?:\s+\d{4})?|Regulations?\b|Directive\b|Ordinance\b|Statute\b|Treaty\b|Convention\b|Amendment\b)'
)

def _noun_ents(text:str, limit:int=40) -> L:
    'ORG, PERSON and LAW entities via regex — 89% of spaCy recall, zero model weight.'
    seen, out = set(), L()
    def _add(t, label):
        n = re.sub(r'\s+', ' ', t.strip())[:60]
        k = n.lower()
        if n and k not in seen: seen.add(k); out.append((n, label))
    for m in _ORG_SUFF.finditer(text): _add(m.group(0).strip(), 'ORG')
    for m in _PERSON_HON.finditer(text): _add(m.group(1), 'PERSON')
    for m in _LAW_INST.finditer(text): _add(m.group(1).strip(), 'LAW')
    return out[:limit]

def signals(text:str,            # the document text
            ner:bool=True,       # run entity extraction (noun entities + keyphrases)
            limit:int=20,        # entities kept
) -> AttrDict:
    'Countable evidence in one document: `counts` per signal, plus the entities it names.'
    counts = {k: len(p.findall(text or '')) for k, p in _SIG.items()}
    noun = L(_noun_ents((text or '')[:NER_CHARS]) if ner else ()).map(
        lambda t: AttrDict(label=t[1], text=t[0]))
    kws  = L(text_entities((text or '')[:NER_CHARS]) if ner else ()).map(
        lambda t: AttrDict(label=(t[1] or 'KEYPHRASE').upper(), text=re.sub(r'\s+', ' ', t[0].strip())[:60])
    ).filter(lambda e: e.text and e.label not in {'ORG', 'PERSON', 'LAW', 'PRODUCT', 'GPE'})
    found = noun + kws
    labels = dict(Counter(e.label.lower() for e in found))
    # `labels` is counted over everything found and `limit` caps only what is *shown*, because
    # `cue_scores` reads the labels: a display cap that silently moved a score would be a trap
    if noun:
        counts = {k: counts.get(k, 0) + (labels.get(k, 0) if k not in counts else 0)
                  for k in (*counts, *(l for l in labels if l not in ('keyphrase', 'term')))}
    return AttrDict(counts={k: v for k, v in counts.items() if v}, labels=labels,
                    ents=found.sorted(key=lambda e: e.label in ('KEYPHRASE', 'TERM'))[:limit],
                    method=('ner+regex' if noun else 'keyphrase+regex' if kws else 'regex'))

## Doctypes: what a document is

A cue table, scored. `cue_scores` is a pure function of the text, which is what makes it testable —
and `guess_type` reports whether the winner is worth trusting.

`cue_scores` has three legs: the fraction of cue phrases matched, whether the signals the type
*needs* are present at all, and the fraction of expected entity labels seen. That last is read off
`labels` and only scores when the entity extractor found named entities (`method='ner+regex'`); the
leg is dropped and its weight redistributed when no entities were found, which keeps scores
comparable across documents. Keyphrases carry no structural label, so they cannot stand in for it.

`decisive` is the point of `guess_type`: a cue table is right often and cheaply, and *knows when it
is guessing*. A clear winner needs no model; a two-way tie is exactly the case worth spending one
on, and `categorize` reads it to decide whether to call an LLM at all.

In [ ]:
#| export
DOCTYPES = {
    # label: the cue phrases that are evidence for it, the regex signals it needs, and the entity
    # labels it expects. Each leg is scored as a *fraction* matched, never a count, so a long
    # document cannot out-score a short one on the same evidence. The entity labels are drawn from
    # the set `_noun_ents` produces (`ORG`, `PERSON`, `LAW`) — all structurally nameable things,
    # never money or dates, which the regex leg reads far more reliably off an invoice anyway.
    'invoice': dict(needs=('money',), ents=('ORG',), cues=(
        r'\b(?:tax )?invoice\b', r'\b(?:amount|balance|total) due\b', r'\bbill(?:ed)? to\b|\bremit\b',
        r'\bpayment terms?\b|\bnet \d{1,3}\b', r'\bsub-?total\b', r'\b(?:vat|gst|sales tax)\b')),
    'receipt': dict(needs=('money',), ents=('ORG',), cues=(
        r'\breceipt\b', r'\bthank you for your (?:order|purchase|payment)\b',
        r'\bcard\b[^\n]{0,16}\bending\b|\bcash tendered\b|\bchange due\b',
        r'\btransaction (?:id|no)\b|\bauth(?:orisation|orization) code\b',
        r'\bpaid\b|\bpayment received\b', r'\bmerchant\b|\bstore #\s?\d+\b')),
    'purchase_order': dict(needs=('ref',), ents=('ORG',), cues=(
        r'\bpurchase order\b|\bp\.?o\.?\s*(?:no|number|#)', r'\bship(?:[ -]?to)?\b',
        r'\bdelivery date\b|\brequested delivery\b', r'\bvendor\b|\bsupplier\b',
        r'\brequisition\b', r'\bunit price\b')),
    'quote': dict(needs=('money',), ents=('ORG',), cues=(
        r'\bquotation\b|\bquote\s*(?:no|number|#)|\bestimate\b', r'\bvalid (?:until|for|through)\b',
        r'\bunit price\b', r'\bterms and conditions\b',
        r'\bwe are pleased to (?:quote|offer)\b|\bno obligation\b')),
    'catalogue': dict(needs=('money',), ents=(), cues=(
        r'\bcatalog(?:ue)?\b|\bprice list\b|\bproduct list\b',
        r'\bsku\b|\bmodel\s*(?:no|number)\b|\bpart\s*(?:no|number)\b',
        r'\b(?:in|out of) stock\b|\bavailability\b', r'\bper (?:unit|pack|case|kg|litre|liter)\b',
        r'\bspecifications?\b|\bdimensions\b', r'\badd to (?:cart|basket)\b')),
    'contract': dict(needs=(), ents=('ORG', 'LAW'), cues=(
        r'\bagreement\b|\bcontract\b', r'\bparties\b|\bby and between\b',
        r'\bhereby\b|\bwhereas\b|\bhereinafter\b', r'\bshall\b',
        r'\bgoverning law\b|\bjurisdiction\b|\btermination\b',
        r'\bconfidential(?:ity)?\b|\bindemnif|\bliability\b')),
    'resume': dict(needs=('email',), ents=('PERSON', 'ORG'), cues=(
        r'\b(?:curriculum vitae|resum[eé])\b',
        r'\bwork experience\b|\bemployment history\b|\bprofessional experience\b',
        r'\beducation\b', r'\bskills\b', r'\bcertification',
        r'\breferences available\b|linkedin\.com/in/')),
    'paper': dict(needs=('citation',), ents=('ORG', 'PERSON'), cues=(
        r'\babstract\b', r'\bintroduction\b', r'\brelated work\b|\bmethodology\b|\bexperiments?\b',
        r'\breferences\b|\bbibliography\b', r'\bwe (?:propose|present|show|evaluate)\b',
        r'\bdoi:|\barxiv:')),
    'report': dict(needs=(), ents=('ORG',), cues=(
        r'\bexecutive summary\b', r'\bfindings\b|\bconclusions?\b|\brecommendations?\b',
        r'\bq[1-4] \d{4}\b|\bfiscal year\b|\bfy\d{2}\b|\bquarter\b',
        r'\btable \d+\b|\bfigure \d+\b|\bappendix\b',
        r'\byear[- ]on[- ]year\b|\brevenue\b|\bmargin\b')),
    'documentation': dict(needs=('heading',), ents=(), cues=(
        r'\binstallation\b|\bgetting started\b|\bquick ?start\b',
        r'\busage\b|\bexamples?\b|\bapi reference\b', r'\bparameters?\b|\breturns\b|\barguments?\b',
        r'\bpip install\b|\bnpm install\b|\bdocker run\b', r'\bconfiguration\b|\bsee also\b',
        r'^```')),
    'code': dict(needs=('code',), ents=(), cues=(
        r'^\s*(?:def|class)\s+\w+|^\s*(?:function|const|let|var)\s+\w+',
        r'^\s*(?:import|from|#include|package|use)\s+\w', r'\breturn\b',
        r'^\s*(?://|#|/\*)', r'\b(?:if|for|while)\s*\(', r'\b(?:public|private|static|async)\b')),
    'email': dict(needs=('email',), ents=('PERSON',), cues=(
        r'^(?:from|to|cc|bcc|subject|sent):', r'^\s*(?:dear|hi|hello)\b',
        r'\b(?:best|kind) regards\b|\bsincerely\b|\bthanks,\b',
        r'\bforwarded message\b|\bwrote:\s*$', r'\bunsubscribe\b')),
    'meeting_notes': dict(needs=(), ents=('PERSON',), cues=(
        r'\b(?:meeting|call) notes\b|\bminutes\b', r'\battendees\b|\bparticipants\b|\bpresent\b',
        r'\bagenda\b', r'\baction items?\b|\bnext steps\b|\bfollow[- ]ups?\b',
        r'\bdecided\b|\bdecisions?\b|\bowner\b|\bdue by\b')),
    'form': dict(needs=('blank',), ents=(), cues=(
        r'\bplease (?:complete|fill|print|sign)\b', r'\bfor office use only\b',
        r'\bapplicant\b|\bapplication (?:form|for)\b', r'\bsignature\b|\bdate signed\b',
        r'\b(?:full name|surname|given names?|date of birth|dob)\b', r'_{4,}')),
    'transcript': dict(needs=('time',), ents=('PERSON',), cues=(
        r'\btranscript\b', r'^\s*\[?\d{1,2}:\d{2}', r'^\s*(?:speaker \d|[A-Z][a-z]+\s?[A-Z]?[a-z]*):',
        r'\b(?:um|uh|you know|i mean)\b', r'\bwelcome (?:back|to)\b|\bthanks for (?:watching|listening)\b')),
    'article': dict(needs=(), ents=('PERSON', 'ORG'), cues=(
        r'\bpublished\b|\bposted (?:on|by)\b', r'\bread more\b|\bshare this\b|\bcomments?\b',
        r'\bsubscribe\b|\bnewsletter\b', r'\baccording to\b', r'\btags?:|\bcategor(?:y|ies):')),
}
_CUES = {l: [re.compile(c, re.I|re.M) for c in d['cues']] for l, d in DOCTYPES.items()}

# What acquired a document is evidence about what it is, and better evidence than any cue phrase:
# a YouTube ingest *is* a transcript. Only the certain cases are listed.
KIND_HINT = dict(youtube='transcript', arxiv='paper', code='code')
MIN_SCORE, MIN_MARGIN, KIND_BONUS = 0.4, 0.12, 0.2

def cue_scores(text:str, sig=None) -> dict:
    'Every doctype scored against `text`, best first — the categorisation no model is needed for.'
    sig = sig if sig is not None else signals(text)
    ner, out = sig.method.startswith('ner'), {}
    for lbl, d in DOCTYPES.items():
        cues = [p for p in _CUES[lbl] if p.search(text or '')]
        need = [s for s in d['needs'] if sig.counts.get(s)]
        ents = [e for e in d['ents'] if sig.labels.get(e.lower())] if ner else []
        legs = [(0.6, len(cues)/len(_CUES[lbl]))]
        if d['needs']: legs.append((0.25, len(need)/len(d['needs'])))
        if ner and d['ents']: legs.append((0.15, len(ents)/len(d['ents'])))
        w = sum(x for x, _ in legs)
        out[lbl] = round(sum(x*f for x, f in legs)/w, 3)
    return dict(sorted(out.items(), key=lambda kv: -kv[1]))

def guess_type(text:str,             # the document text
               sig=None,             # a signals() result, computed if not given
               kind:str=None,        # the vault kind that acquired it, if known
               min_score:float=MIN_SCORE,    # below this the cues have not decided
               min_margin:float=MIN_MARGIN,  # runner-up this close means they have not either
) -> AttrDict:
    'The best doctype for `text` from cues alone, and whether that guess is worth trusting.'
    sig = sig if sig is not None else signals(text)
    sc = cue_scores(text, sig)
    if kind in KIND_HINT: sc[KIND_HINT[kind]] = round(min(1.0, sc[KIND_HINT[kind]] + KIND_BONUS), 3)
    ranked = sorted(sc.items(), key=lambda kv: -kv[1])
    (top, best), (_, second) = ranked[0], (ranked[1] if len(ranked) > 1 else ('', 0.0))
    return AttrDict(doctype=top if best else 'other', score=best, margin=round(best-second, 3),
                    decisive=best >= min_score and best-second >= min_margin,
                    scores=dict(ranked[:5]), method=sig.method)

`categorize` puts the two together, and writes the verdict into the document's `meta` so the vault
remembers it.

Cues first, a model only if they cannot decide. That order is the design, not an optimisation:
typing ten thousand documents through an LLM is hours of compute to answer a question a regex table
answers for most of them, and the cases the table gets wrong are the ones where it scores two types
nearly the same — precisely when `llm=None` spends a model call. `llm` is a word rather than a flag
because it has three states and a `bool` only reaches two of them from a command line. `by` records
which leg decided, so a vault's types can be audited and re-run selectively.

`categorize_all` records one failure and steps over it rather than raising: a run over a whole vault
must not be lost to a single unreadable document, and `force=False` makes it cheap to re-run after
an ingest.

`reshelf` is the route that had to wait. Acquisition routes on what it can tell at the door, and for
most things that is enough — an arXiv id is a paper. A PDF is as likely an invoice as a paper and
nothing at the door can say which, so the cheap route runs at ingest and this one runs afterwards.
A move is a re-ingest: the other shelf has a different encoder, so the vectors are made again, and
the text is the reassembled document rather than the original file — page boundaries do not survive
it. Re-`grab` the source when the pages matter.

`ner` points the same extractor at a single document. It is not a replacement for `connect()`, which
builds one graph over the whole corpus so sections can reach each other; this answers the narrower
question of what *this* document talks about, with no index to rebuild.

In [ ]:
#| export
TYPE_SP = """You label one document with what kind of thing it is.

Judge the document as a whole, by its purpose, not by a word that happens to appear in it: a paper
about invoicing is a paper. Reply with exactly one label from the list and nothing else."""

def model_cached(mid:str) -> bool:
    'Is this model already in the Hugging Face cache? A cache scan, never a download.'
    try:
        from huggingface_hub import scan_cache_dir
        return any(r.repo_id == mid for r in scan_cache_dir().repos)
    except Exception: return False

@patch
def categorize(self:Vault,
               ref,                # doc_id, source, title, a path on disk, or a loaded `document()`
               model:str=None,     # an id, a path, `mlx/…`; None -> $VISHALAKSHI_MODEL
               chat_kw:dict=None,  # anything else rishi's `Chat` takes: temp, runtime, think, …
               llm:str='auto',     # 'auto' -> only when the cues cannot decide | 'always' | 'never'
               labels:str=None,    # comma-separated labels to choose from; None -> DOCTYPES
               max_chars:int=6000, # chars of the document the model and the cues see
               ner:bool=True,      # run entity extraction alongside the regex signals
               save:bool=True,     # write the verdict into the document's meta
) -> AttrDict:
    'What kind of document this is: invoice, catalogue, contract, paper, transcript, code…'
    d = (ref if isinstance(ref, dict) and 'text' in ref
         else self.document(ref, max_chars=max(max_chars, 4000)))
    txt = (d.text or '')[:max_chars]
    if not txt.strip():
        return AttrDict(doc_id=d.doc_id, title=d.title, doctype=None, skipped='no text to judge')
    sig = signals(txt, ner=ner)
    g = guess_type(txt, sig, kind=d.kind)
    dt, by = g.doctype, f'cues ({g.method})'
    mode = {True: 'always', False: 'never', None: 'auto'}.get(llm, str(llm).lower())
    assert mode in ('auto', 'always', 'never'), f"llm must be auto, always or never — not {llm!r}"
    if mode == 'always' or (mode == 'auto' and not g.decisive):
        lbls = L(labels.split(',') if isinstance(labels, str) else labels or list(DOCTYPES)).map(str.strip)
        mid = model or dflt_model
        try:
            # `auto` is allowed to *use* a model, not to go and fetch one: a vault of ten thousand
            # documents must not turn a one-line note into a multi-gigabyte download. rishi decides
            # what runs the id; whether the weights are already here is the one thing it cannot say.
            if (mode == 'auto' and is_stock_chat()
                    and infer_runtime(mid) != 'remote' and not model_cached(mid)):
                by = f'cues ({g.method}); {mid or "no model"} is not downloaded, so none was asked'
            else:
                said = new_chat(mid, **(chat_kw or {})).classify(f'{d.title}\n\n{txt}', list(lbls)+['other'], sp=TYPE_SP)
                if said in lbls or said == 'other': dt, by = said, f'llm ({mid or "rishi default"})'
                else: by = f'cues ({g.method}); llm answered {said[:40]!r}, not a label'
        except Exception as e:
            # `auto` asked for a model *if one can be had*. There may be no weights and no network,
            # and a corpus 90% typed by cues alone is worth far more than a traceback — so the cue
            # verdict stands, and says why. `always` was an instruction, and raises.
            if mode == 'always': raise
            by = f'cues ({g.method}); no model available ({type(e).__name__})'
    res = AttrDict(doc_id=d.doc_id, title=d.title, kind=d.kind, doctype=dt, score=g.score,
                   margin=g.margin, decisive=g.decisive, by=by, scores=g.scores,
                   signals=sig.counts, ents=sig.ents, saved=False)
    if save and d.doc_id:
        self.set_meta(d.doc_id, doctype=dt, doctype_by=by, doctype_score=g.score)
        res.saved = True
    return res

@patch
def categorize_all(self:Vault,
                   kind:str=None,     # restrict to one or more KINDS
                   force:bool=False,  # re-type documents that already carry a doctype
                   limit:int=None,    # stop after this many
                   **kw               # forwarded to categorize (model=, chat_kw=, llm=, ner=, max_chars=)
) -> AttrDict:
    'Type every document in the vault that is not typed yet, and report the shape of the corpus.'
    docs = self.sources(kind)
    if not force: docs = docs.filter(lambda r: not (r['meta'] or {}).get('doctype'))
    out = L()
    for r in docs[:limit]:
        try: out.append(self.categorize(r['id'], **kw))
        except Exception as e:
            out.append(AttrDict(doc_id=r['id'], title=r['title'], doctype=None,
                                error=f'{type(e).__name__}: {str(e)[:200]}'))
    return AttrDict(n=len(out), by_type=dict(Counter(o.doctype for o in out).most_common()),
                    errors=out.filter(lambda o: o.get('error')), results=out)

@patch
def doctypes(self:Vault, kind:str=None) -> dict:
    'How many documents of each type the vault holds — `untyped` counts the ones never categorised.'
    c = Counter((r['meta'] or {}).get('doctype') or 'untyped' for r in self.sources(kind))
    return dict(c.most_common())

@patch
def of_type(self:Vault, doctype:str, kind:str=None) -> L:
    'Every document categorised as `doctype`, newest first.'
    return self.sources(kind).filter(lambda r: (r['meta'] or {}).get('doctype') == doctype)

# What a document turned out to *be* — as opposed to how it arrived, which is what `KIND_SHELF`
# covers. Only types whose shelf is not the default: everything else is already where it belongs.
DOCTYPE_SHELF = {'paper': 'papers', 'code': 'code', 'catalogue': 'data'}

@patch
def reshelf(self:Vault,
            ref,                      # doc_id, source, or a title substring
            shelf:str=None,           # where to put it; None -> whatever its doctype says
            llm:str='auto',           # the LLM leg of the categorisation that picks the shelf
            max_chars:int=2_000_000,  # cap on the text carried over
            **kw                      # forwarded to categorize (model=, chat_kw=, ner=)
) -> AttrDict:
    'Move a document to the shelf its *type* says it belongs on — the route that had to wait.'
    d = self.document(ref, max_chars=max_chars)
    c = self.categorize(d, save=False, llm=llm, **kw) if shelf is None else None
    nm = shelf or DOCTYPE_SHELF.get(c.doctype)
    out = AttrDict(doc_id=d.doc_id, title=d.title, doctype=c.doctype if c else None,
                   was=self.name, store=self.name, moved=False)
    if not nm or nm == self.name or not d.doc_id: return out
    r = self.shelf(nm).add(d.text, d.title, source=d.source, kind=d.kind, force=True,
                           meta=dict(d.meta, doctype=out.doctype, reshelved_from=self.name))
    self.forget(d.doc_id)
    return AttrDict(out, doc_id=r.get('doc_id'), store=nm, moved=True, chunks=r.get('chunks'))

@patch
def ner(self:Vault,
        ref:str,              # doc_id, source, title substring, or a path on disk
        limit:int=40,         # entities returned
        max_chars:int=20000,
) -> AttrDict:
    'What one document names: organisations, people, places, products — and the terms around them.'
    d = self.document(ref, max_chars=max_chars)
    sig = signals(d.text, limit=limit)
    return AttrDict(doc_id=d.doc_id, title=d.title, method=sig.method, counts=sig.counts,
                    labels=sig.labels, ents=sig.ents)

## Schemas: the shape of an extraction

The shapes worth having ready, and `dyn_schema` for the ones that are not — a dataclass built from
`'vendor:str, total:float, items:dicts'` at the moment you ask for it.

`dyn_schema` is what makes a structured response *dynamic*: the caller describes the fields they
want in the question itself and the model is constrained to them, with no class declared first.
Every field defaults, so a document that does not say something yields a blank rather than an
exception. `list` fields default to `None` rather than `[]` deliberately — a `default_factory`
sentinel is not JSON-serialisable, and it is the *schema* that is shipped to the model, so `extract`
turns the `None`s back into empty lists on the way out. An unknown type name becomes `str` on the
same reasoning: a field read as text is recoverable, an exception raised halfway through a batch is
not.

The docstring on each schema below is not documentation — it is shipped to the model as the
description of the shape it must fill, so it is written for the model to read.

In [ ]:
#| export
@dataclass
class Invoice:
    """An invoice, purchase order or quotation.

    items: one entry per line, each {description, qty, unit_price, amount}.
    Amounts are bare numbers with the currency in `currency`; dates are ISO (2024-03-01)."""
    number:str = ''
    date:str = ''
    due_date:str = ''
    vendor:str = ''
    vendor_tax_id:str = ''
    bill_to:str = ''
    ship_to:str = ''
    currency:str = ''
    subtotal:float = 0
    tax:float = 0
    total:float = 0
    payment_terms:str = ''
    items:list[dict] = None

@dataclass
class Receipt:
    """A receipt for a completed payment.

    items: one entry per line, each {description, qty, amount}. `paid_with` is the tender or the
    last digits of the card. Dates are ISO (2024-03-01)."""
    merchant:str = ''
    date:str = ''
    transaction_id:str = ''
    currency:str = ''
    subtotal:float = 0
    tax:float = 0
    total:float = 0
    paid_with:str = ''
    items:list[dict] = None

@dataclass
class Catalogue:
    """A product catalogue, price list or listing page.

    products: one entry per product, each {sku, name, category, price, currency, availability, url}.
    Prices are bare numbers."""
    name:str = ''
    vendor:str = ''
    currency:str = ''
    updated:str = ''
    n_products:int = 0
    products:list[dict] = None

@dataclass
class Contract:
    """An agreement between parties.

    parties: the legal names. obligations: what each side must do. Dates are ISO (2024-03-01)."""
    title:str = ''
    parties:list[str] = None
    effective_date:str = ''
    end_date:str = ''
    term:str = ''
    value:str = ''
    governing_law:str = ''
    termination:str = ''
    obligations:list[str] = None

@dataclass
class Resume:
    """One person's CV.

    experience: one entry per role, each {employer, title, start, end, summary}.
    education: one entry per qualification, each {institution, qualification, year}."""
    name:str = ''
    email:str = ''
    phone:str = ''
    location:str = ''
    headline:str = ''
    years_experience:float = 0
    skills:list[str] = None
    experience:list[dict] = None
    education:list[dict] = None

@dataclass
class Paper:
    """An academic paper. authors: names in order. findings: the claims the paper actually makes."""
    title:str = ''
    authors:list[str] = None
    venue:str = ''
    year:str = ''
    doi:str = ''
    abstract:str = ''
    method:str = ''
    findings:list[str] = None
    limitations:list[str] = None

@dataclass
class MeetingNotes:
    """Notes from one meeting. actions: one entry per commitment, each {what, owner, due}."""
    title:str = ''
    date:str = ''
    attendees:list[str] = None
    topics:list[str] = None
    decisions:list[str] = None
    actions:list[dict] = None

@dataclass
class Summary:
    """What one document says, when no more specific shape fits.

    entities: the organisations, people and places it names. dates: ISO where the document allows."""
    title:str = ''
    doctype:str = ''
    about:str = ''
    key_points:list[str] = None
    entities:list[str] = None
    dates:list[str] = None
    numbers:list[str] = None
    open_questions:list[str] = None

# A purchase order and a quotation carry an invoice's fields under other names, so they share its
# schema rather than getting a near-duplicate of it.
SCHEMAS = dict(invoice=Invoice, purchase_order=Invoice, quote=Invoice, receipt=Receipt,
               catalogue=Catalogue, contract=Contract, resume=Resume, paper=Paper,
               meeting_notes=MeetingNotes, other=Summary)

FIELD_TYPES = {'str':str, 'text':str, 'int':int, 'float':float, 'number':float, 'num':float, 'bool':bool,
               'list':list, 'dict':dict, 'strs':list[str], 'dicts':list[dict],
               'list[str]':list[str], 'list[dict]':list[dict]}
_DFLT = {str: '', int: 0, float: 0.0, bool: False}

def dyn_schema(spec,                    # 'vendor:str, total:float, items:dicts', or {'vendor':'str'}, or ['vendor']
               name:str='Extracted',    # class name, which the model sees
               doc:str=None,            # class docstring, which the model also sees
) -> type:
    'A dataclass built at runtime from a field spec — the shape of an answer, named in one string.'
    if isinstance(spec, str): spec = [p for p in re.split(r'[,\n]', spec) if p.strip()]
    if isinstance(spec, dict): spec = [f'{k}:{v}' for k, v in spec.items()]
    flds = []
    for p in spec:
        nm, _, ty = (p if isinstance(p, str) else ':'.join(p)).partition(':')
        t = FIELD_TYPES.get(ty.strip().lower() or 'str', str)
        flds.append((re.sub(r'\W', '_', nm.strip()), t, _DFLT.get(t, None)))
    if not flds: raise ValueError(f'no fields in schema spec {spec!r}')
    return make_dataclass(re.sub(r'\W', '', name) or 'Extracted', flds,
                          namespace=dict(__doc__=doc or 'The fields to pull out of the document.'))

def as_schema(spec, name:str='Extracted', doc:str=None) -> type:
    'Whatever names a shape, as a dataclass: a `SCHEMAS` key, a dataclass, or a `dyn_schema` spec.'
    if is_dataclass(spec): return spec
    if isinstance(spec, str) and spec.strip() in SCHEMAS: return SCHEMAS[spec.strip()]
    return dyn_schema(spec, name=name, doc=doc)

def _norm(obj, schema) -> dict:
    "A structured reply as a plain dict, with the `None` a list field defaults to turned back into `[]`."
    d = asdict(obj) if is_dataclass(obj) and not isinstance(obj, type) else dict(obj or {})
    lists = {f.name for f in fields(schema) if f.type is list or get_origin(f.type) is list}
    return {k: ([] if v is None and k in lists else v) for k, v in d.items()}

def schema_str(schema) -> str:
    "A schema written out for a model to read: its docstring, then `name: type` per field."
    def ty(t):
        if isinstance(t, str): return t
        # `list[dict].__name__` is just 'list' — the parameter is the half the model needs
        return str(t).replace('typing.', '') if get_origin(t) else getattr(t, '__name__', str(t))
    return ((schema.__doc__ or '').strip() + '\n\n{\n'
            + ',\n'.join(f'  "{f.name}": {ty(f.type)}' for f in fields(schema)) + '\n}')

def _json_reply(ch, prompt:str, schema, sp:str='') -> object:
    'Ask for the shape as a JSON object in prose and rebuild it, with the model left unconstrained.'
    ch.hist = []
    txt = resp_text(ch(f'{sp}\n\n{prompt}\n\nReply with only a JSON object in a ```json fence, with '
                       f'exactly these keys and types:\n\n{schema_str(schema)}'))
    d = json.loads(extract_fence(split_reasoning(txt)[0], 'json'))
    nms = {f.name for f in fields(schema)}
    return schema(**{k: v for k, v in d.items() if k in nms})

def structured(ch,                # a `rishi.Chat`, e.g. from `new_chat`
               prompt:str,        # the document and the instruction
               schema,            # the dataclass the reply must fill
               sp:str='',         # system prompt for the extraction
) -> dict:
    'A structured reply as a dict — retried as a plain JSON reply when the constrained call fails.'
    try: return _norm(ch.structured(prompt, schema, sp=sp), schema)
    except Exception as e:
        warnings.warn(f'{type(e).__name__} on a constrained call for {schema.__name__} '
                      f'({str(e)[:150]}) — retrying as a JSON reply.')
        return _norm(_json_reply(ch, prompt, schema, sp), schema)

## Extraction

One document in, a dict of fields out. With no `schema`, the document is categorised first and the
shape follows from what it turned out to be. That is the whole point of typing a corpus: point this
at a folder of mixed paperwork and each document is read against the shape that fits it. rishi
constrains the model to the schema — a forced tool call on the hosted and LiteRT backends, a grammar
on llama.cpp, a parsed JSON reply on MLX — so what comes back is a dict with the fields you asked
for, not prose about them.

`structured` is there because the constrained path is the *good* path, not a guaranteed one. A small
local model asked for a nested field like an invoice's `items` can emit a tool call its own runtime
cannot parse — LiteRT reports that as an opaque `send_message failed`, and a 2GB model load plus a
minute of CPU is a bad thing to lose to a syntax error. So the constrained call is tried first and,
when it fails, the same document is asked again as a plain JSON reply and parsed. Unknown keys are
dropped and missing ones keep their default, so the second leg cannot return a shape the first would
not have. It warns when it falls back: which leg answered is worth knowing, because the JSON leg is
unconstrained and can drift.

`rows` is the point of `extract_all`: every result flattened to `doc_id` and `doc_title` plus the
schema's own fields, which is a dataframe or a CSV away from being useful. The identity columns
carry the `doc_` prefix because `title` is a field of half the schemas, and neither title should
shadow the other. A document that yields nothing is recorded in `errors` and the run continues.

In [ ]:
#| export
EXTRACT_SP = """You pull structured fields out of one document.

Rules:
- Copy what the document states. Do not compute, convert, round or tidy a value.
- A field the document does not state stays empty. An invented number is worse than a blank one.
- Numbers are bare: 1240.50, not "$1,240.50". The currency belongs in its own field.
- Dates are ISO: 2024-03-01.
- A list field takes every entry the document has, in the order it has them."""

@patch
def extract(self:Vault,
            ref:str,               # doc_id, source, title substring, or a path on disk
            schema=None,           # a SCHEMAS key, a dataclass, or a 'field:type, …' spec; None -> from the doctype
            model:str=None,        # an id, a path, `mlx/…`; None -> $VISHALAKSHI_MODEL
            chat_kw:dict=None,     # anything else rishi's `Chat` takes: temp, runtime, think, …
            max_chars:int=8000,    # chars of the document the model sees
            sp:str=EXTRACT_SP,     # system prompt for the extraction
            save:bool=False,       # write the fields into the document's meta
            llm:str='auto',        # the LLM leg of the categorisation, when picking the schema
) -> AttrDict:
    "Pull structured fields out of one document — an invoice's totals, a catalogue's products."
    d = self.document(ref, max_chars=max_chars)
    if not (d.text or '').strip():
        return AttrDict(doc_id=d.doc_id, title=d.title, source=d.source, doctype=None, schema=None,
                        fields={}, skipped='no text to extract from')
    dt, cat = None, None
    if schema is None:
        cat = self.categorize(d, model=model, chat_kw=chat_kw, llm=llm, save=save)
        dt = cat.doctype
        sch = SCHEMAS.get(dt, Summary)
    else: sch = as_schema(schema)
    prompt = (f'{d.title}\n(source: {d.source})\n\n{d.text}\n\n---\n\n'
              f'Pull the fields of `{sch.__name__}` out of the document above.')
    ch = new_chat(model, **(chat_kw or {}))
    flds = structured(ch, prompt, sch, sp=sp)
    if save and d.doc_id: self.set_meta(d.doc_id, extracted=flds, extracted_as=sch.__name__)
    return AttrDict(doc_id=d.doc_id, title=d.title, source=d.source, doctype=dt, schema=sch.__name__,
                    fields=flds, chars=len(d.text), truncated=d.truncated, model=model or dflt_model,
                    runtime=ch.runtime, categorized=cat, usage=getattr(ch, 'use', None))

@patch
def extract_all(self:Vault,
                doctype:str=None,   # only documents already categorised as this
                kind:str=None,      # only these KINDS
                schema=None,        # one shape for all of them; None -> each document's own doctype
                limit:int=None,     # stop after this many
                **kw                # forwarded to extract (model=, chat_kw=, max_chars=, save=, sp=)
) -> AttrDict:
    'Extract from many documents at once — a folder of invoices as one table.'
    docs = self.of_type(doctype, kind) if doctype else self.sources(kind)
    rows, errs = L(), L()
    for r in docs[:limit]:
        try:
            e = self.extract(r['id'], schema=schema, **kw)
            if e.get('skipped'): errs.append(dict(doc_id=r['id'], title=r['title'], error=e.skipped))
            else: rows.append(dict({'doc_id': e.doc_id, 'doc_title': e.title, 'schema': e.schema},
                                   **e.fields))
        except Exception as ex:
            errs.append(dict(doc_id=r['id'], title=r['title'], error=f'{type(ex).__name__}: {str(ex)[:200]}'))
    return AttrDict(n=len(rows), doctype=doctype, schema=schema, rows=rows, errors=errs)

## Asking over one whole document

`ask_doc` is `ask` with `ref=` filled in, and it is kept because the name says what it does: start
from a document you have already chosen, read *all* of it, and add the rest of the vault behind it.
The document is section `[1]`, so the citation contract is the one `ask` already keeps — `[n]`
resolves through `cited` to something you can `read()`. Pass `schema=` and the reply is a dict of the
fields you named instead of prose. The document need not be in the vault at all: a path on disk is
read straight off it.

In [ ]:
#| export
@patch
def ask_doc(self:Vault,
            ref:str,              # doc_id, source, title substring, or a path on disk
            question:str,         # what you want to know about it
            schema=None,          # answer as this shape instead of prose
            max_chars:int=8000,   # chars of the document the model sees
            related:int=3,        # sections from the *rest* of the vault to add as context
            **kw                  # forwarded to `ask` (model=, chat_kw=, sp=)
) -> AttrDict:
    'Answer a question about one whole document — `ask` with that document as section [1].'
    return self.ask(question, ref=ref, schema=schema, doc_chars=max_chars, related=related, **kw)

## Try it

Two documents, one of each kind, and nothing here needs a model yet — the cue table, the signals and
the schema machinery are the legs that run everywhere.

In [ ]:
#| eval:false
from vishalakshi import Vault

INVOICE = '''# INVOICE

Invoice No: ACM-2024-0117
Date: 2024-03-01
Payment terms: Net 30

Bill to: Contoso GmbH, Berlin
From: Acme Supplies Ltd

## Line items

| Description | Qty | Unit price | Amount |
|---|---|---|---|
| Widget, steel | 12 | $8.50 | $102.00 |
| Gasket, nitrile | 40 | $1.20 | $48.00 |

Subtotal: $150.00
VAT (20%): $30.00
Total due: $180.00
'''

CATALOGUE = '''# Spring price list

## Widget, steel

SKU: WS-100. Price: $8.50 per unit. In stock.
Specifications: 40mm, zinc plated. Dimensions 40x12x4mm.

## Gasket, nitrile

SKU: GN-220. Price: $1.20 per unit. Out of stock.
Add to cart to be notified.
'''

v = Vault(':memory:')
v.add(INVOICE, 'Acme invoice ACM-2024-0117', source='/inbox/acme-0117.md')
v.add(CATALOGUE, 'Spring price list', source='/inbox/spring-list.md')
v.doctypes()

/Users/71293/code/personal/orgs/vishalakshi/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'untyped': 2}

In [ ]:
#| eval: false
r = v.categorize('Acme invoice ACM-2024-0117', llm='never')
r.doctype, r.score, r.decisive, r.by

('invoice', 1.0, True, 'cues (ner+regex)')

In [ ]:
#| eval: false
test_eq(r.doctype, 'invoice')
assert r.decisive, r.scores          # the cues are sure enough that no model is needed
test_eq(v.doc(r.doc_id)['meta']['doctype'], 'invoice')   # written back into the vault
test_eq(v.categorize('Spring price list', llm='never').doctype, 'catalogue')
test_eq(v.doctypes(), {'invoice': 1, 'catalogue': 1})
test_eq(v.of_type('invoice').attrgot('title'), ['Acme invoice ACM-2024-0117'])

In [ ]:
#| eval:false
#| hide
# the regex leg owns the numeric evidence and runs regardless of whether entities are found
sig = signals(INVOICE)
assert sig.method in ('ner+regex', 'keyphrase+regex', 'regex'), sig.method
for k in ('money', 'date', 'ref', 'tax', 'percent', 'table', 'heading'): assert sig.counts.get(k), k
test_eq(signals(INVOICE, ner=False).method, 'regex')
test_eq(signals(INVOICE, ner=False).ents, [])

# the handrolled entity leg finds company names by legal suffix
if sig.method == 'ner+regex':
    assert any('Acme' in e.text or 'Contoso' in e.text for e in sig.ents), sig.ents
    assert sig.labels.get('org'), sig.labels
    # labelled entities sort before keyphrases
    assert all(e.label not in ('KEYPHRASE', 'TERM') for e in sig.ents[:len(sig.ents.filter(lambda e: e.label not in ('KEYPHRASE','TERM')))])

# `limit` caps the list, never the evidence: a score must not move with it
test_eq(cue_scores(INVOICE, signals(INVOICE, limit=2))['invoice'], cue_scores(INVOICE, sig)['invoice'])
test_eq(len(signals(INVOICE, limit=2).ents), 2)

# a score is a fraction, never a count: the same evidence twice over must not score higher
test_eq(cue_scores(INVOICE)['invoice'], cue_scores(INVOICE + INVOICE)['invoice'])
# and what acquired a document is evidence about what it is
test_eq(guess_type('Um, so, welcome back. 00:12 speaker 1: right.', kind='youtube').doctype, 'transcript')

# a document with nothing to go on says so rather than picking a type at random
g = guess_type('The cat sat on the mat.')
assert not g.decisive and g.score < MIN_SCORE, g
test_eq(v.categorize(AttrDict(doc_id=None, title='t', kind=None, text='   '),
                     llm='never').skipped, 'no text to judge')
# three states, and a word for each: a `bool` cannot carry the third past argparse
test_fail(lambda: v.categorize('Acme invoice', llm='sometimes'), contains='auto, always or never')

In [ ]:
#| hide
# a dynamic schema is the point of `schema=` — the caller names the shape in one string
S = as_schema('vendor:str, total:float, paid:bool, items:dicts')
test_eq([f.name for f in fields(S)], ['vendor', 'total', 'paid', 'items'])
test_eq(asdict(S()), dict(vendor='', total=0.0, paid=False, items=None))
test_eq(_norm(S(), S)['items'], [])            # a list field comes back empty, not None
test_eq(as_schema('invoice'), Invoice)         # a registry key
test_eq(as_schema(Invoice), Invoice)           # a dataclass passes through
test_eq(as_schema({'sku': 'str'}).__name__, 'Extracted')
test_eq([f.name for f in fields(as_schema(['a', 'b']))], ['a', 'b'])
test_fail(lambda: as_schema(''), contains='no fields')

# every schema shipped to a model must be JSON-serialisable, which is why no field uses
# `default_factory` — its sentinel is not
from fastcore.funccall import get_schema
for nm, s in SCHEMAS.items(): assert json.dumps(get_schema(s)['input_schema']), nm

# The JSON-reply leg writes the shape out itself rather than through `get_schema`. `get_schema` reads
# docments, which need *source*, and a `make_dataclass` schema has none — outside a notebook it raises
# `KeyError: '__firstlineno__'`, which is exactly where the fallback would run.
assert '"total": float' in schema_str(S) and '"items": list[dict]' in schema_str(S), schema_str(S)
assert 'items: one entry per line' in schema_str(Invoice)      # the docstring is written for the model


The legs above run everywhere. The two that need a model — the label when the cues cannot decide,
and every structured extraction — are tested against a *real* model rather than a stub, and the
replies are replayed from the `chatcache` directory beside this notebook. A stub would have to
invent what a 2B model says, and inventing it is what hid the failure below: gemma-4-E2B, asked for
an invoice's nested `items`, emits a tool call LiteRT then refuses to parse. That refusal is in the
cache, so the fallback that recovers from it is tested rather than assumed.

Recording is `VISHALAKSHI_RECORD_CHAT=1` with the weights present; without it a miss raises, so CI
can only replay.

In [ ]:
#| hide
from functools import partial
from fastcore.all import Path
from vishalakshi.ask import CachedChat, use_chat

# the recorded replies are keyed on this id, and reading it off `rishi.litert` would make these
# tests need the LiteRT extra — a 20MB wheel, to name a string
GEMMA = 'litert-community/gemma-4-E2B-it-litert-lm'

# nbdev runs a notebook from its own directory; this also works from the repo root
CACHE = (Path('nbs') if Path('nbs').exists() else Path('.'))/'chatcache'
# there is no `chat=` argument any more, so the recorded replies are swapped in for the rest of
# the notebook through `use_chat`. Entered by hand rather than with `with`, because the tests it
# covers run in the cells below this one.
_replay = use_chat(partial(CachedChat, path=CACHE)); _replay.__enter__()

INV = '''# INVOICE

Invoice No: ACM-2024-0117
Date: 2024-03-01
Payment terms: Net 30

Bill to: Contoso GmbH, Berlin
From: Acme Supplies Ltd

## Line items

| Description | Qty | Unit price | Amount |
|---|---|---|---|
| Widget, steel | 12 | $8.50 | $102.00 |
| Gasket, nitrile | 40 | $1.20 | $48.00 |

Subtotal: $150.00
VAT (20%): $30.00
Total due: $180.00
'''
# offline: the hashing encoder is deterministic, so a retrieved prompt is byte-identical on replay
_x = Vault(':memory:', offline=True)
_x.add(INV, 'Acme invoice ACM-2024-0117', source='/inbox/acme-0117.md')
_x.add('   ', 'empty one', source='/inbox/empty.md')
_m = dict(model=GEMMA)


In [ ]:
#| hide
# the LLM leg of categorisation, against what gemma-4-E2B actually replied
c = _x.categorize('/inbox/acme-0117.md', llm='always', save=False, **_m)
assert c.by.startswith('llm'), c.by
assert 'Total due' in _x.document('/inbox/acme-0117.md').text     # the model saw the whole document

# `auto` may use a model, never fetch one: an uncached id is declined before a chat is built
test_eq(model_cached('no-such-org/no-such-model'), False)
_und = AttrDict(doc_id=None, title='untitled', kind=None, text='The cat sat on the mat.')
with use_chat(Chat):   # the guard asks whether the *stock* backend would have to fetch weights
    _by = _x.categorize(_und, model='no-such-org/no-such-model', save=False).by
assert 'is not downloaded' in _by, _by
# ...and the cue verdict is what stands when no model answers, with `by` saying so
assert _by.startswith('cues ('), _by


In [ ]:
#| hide
# extraction with no schema: the doctype picks `Invoice`, whose `items` is the nested field that
# makes LiteRT reject its own tool call — so this is also the test of the JSON-reply fallback
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    e = _x.extract('/inbox/acme-0117.md', **_m)
test_eq((e.schema, e.doctype), ('Invoice', 'invoice'))
test_eq(e.fields['total'], 180.0)                       # what the model read off the document
test_eq(e.fields['number'], 'ACM-2024-0117')
test_eq(len(e.fields['items']), 2)                      # the nested list the fallback recovered
test_eq(e.runtime, 'litert')
assert any('retrying as a JSON reply' in str(x.message) for x in w), [str(x.message) for x in w]

# a schema named in one string, and the flat fields a 2B model is reliable on
f = _x.extract('/inbox/acme-0117.md', schema='vendor:str, total:float', **_m).fields
test_eq(sorted(f), ['total', 'vendor'])
test_eq(f['total'], 180.0)
# a document with nothing in it is skipped before any model is asked
test_eq(_x.extract('/inbox/empty.md', **_m).skipped, 'no text to extract from')
# across the corpus, one row per document, with identity columns that cannot shadow a schema's own.
# `of_type` reads what was *written*, so the type has to be saved first — the cues suffice for that.
test_eq(_x.categorize('/inbox/acme-0117.md', llm='never').doctype, 'invoice')
b = _x.extract_all(doctype='invoice', schema='vendor:str, total:float', **_m)
test_eq((b.n, len(b.errors)), (1, 0))
test_eq(sorted(b.rows[0]), ['doc_id', 'doc_title', 'schema', 'total', 'vendor'])

# `chat_kw` reaches the rest of the constructor: `CachedChat` takes `path`, and pointing it at an
# empty directory turns the hit above into a miss — which is the proof the kwargs arrived
from tempfile import mkdtemp
with ExceptionExpected(KeyError):
    _x.extract('/inbox/acme-0117.md', schema='vendor:str, total:float',
               chat_kw=dict(path=mkdtemp()), **_m)

/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/ipykernel_50248/1309302621.py:184: UserWarning: KeyError on a constrained call for Extracted ('no recorded reply for litert-community/gemma-4-E2B-it-litert-lm|structured|You pull structured fields out of one document.\n\nRules:\n- Copy what… — ) — retrying as a JSON reply.
  warnings.warn(f'{type(e).__name__} on a constrained call for {schema.__name__} '


In [ ]:
#| hide
# ask_doc: the document is section [1], so `ask`'s citation contract holds — [n] resolves to
# something read() can open
a = _x.ask_doc('/inbox/acme-0117.md', 'what is the total due?', related=2, **_m)
assert a.answer, a
test_eq(a.cited.attrgot('node_id')[0], f'{a.doc_id}#0')
assert 'VAT (20%)' in a.prompt, 'the whole document must reach the model, not a retrieved chunk'
assert a.prompt.endswith('Question: what is the total due?')
# the same question as data instead of prose, through a schema built at the moment of asking
d = _x.ask_doc('/inbox/acme-0117.md', 'what is owed and to whom?',
               schema='amount:float, currency:str, owed_to:str', **_m)
test_eq((d.schema, sorted(d.fields), d.answer), ('Answer', ['amount', 'currency', 'owed_to'], None))
test_eq(d.fields['amount'], 180.0)
# and it is `ask` underneath
test_eq(_x.ask('what is the total due?', ref='/inbox/acme-0117.md', related=2, **_m).answer, a.answer)

# a file the vault has never seen, straight off disk — no model needed to prove where it came from
from fastcore.all import Path
from tempfile import mkdtemp
_p = Path(mkdtemp())/'notes.md'
_p.write_text('# Standup\n\nAttendees: Ana, Bo. Action items: Ana to ship the parser.')
test_eq(_x.document(_p).origin, 'disk')
test_eq(_x.categorize(str(_p), llm='never', save=False).doctype, 'meeting_notes')


In [ ]:
#| eval:false
#| hide
# the route that had to wait: what acquisition could not tell at the door, the doctype can
v.shelf('papers', offline=True)          # pre-registered, so this test pays for no encoder download
r = v.reshelf('Acme invoice', llm='never')
test_eq((r.doctype, r.moved), ('invoice', False))   # an invoice has no shelf of its own: it stays put

PAPER = ('# Late chunking\n\n## Abstract\n\nWe propose contextual chunk embeddings.\n\n'
         '## Introduction\n\nRelated work [1] et al. improved on this.\n\n## References\n\ndoi:10.1/x')
v.add(PAPER, 'late chunking', source='/in/lc.pdf')
r = v.reshelf('/in/lc.pdf', llm='never')
test_eq((r.doctype, r.was, r.store, r.moved), ('paper', 'store', 'papers', True))
test_eq(v.doc('/in/lc.pdf'), None)                                    # gone from the shelf it was on
assert v.shelf('papers').document('/in/lc.pdf').text.startswith('# Late chunking')
test_eq(v.shelf('papers').doc('/in/lc.pdf')['meta']['doctype'], 'paper')   # and it remembers why
test_eq(v.reshelf('Spring price list', shelf='papers').moved, True)        # named: no categorisation

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()